In [1]:
# MODELADO DE PANEL - PARTE 1 (PREPARACIÓN Y TEST DE HAUSMAN)
# Versión corregida con eliminación automática de variables absorbidas

import pandas as pd
import numpy as np
import scipy.stats as stats
from linearmodels.panel import PanelOLS, RandomEffects

# 1. Cargar datos mensuales
df = pd.read_csv('/home/hjvargaso/proyectos/tesis_sequia_2/data/raw/datos_mensuales_con_spei.csv')
df['fecha'] = pd.to_datetime(df['fecha'])

# Diccionario definitivo con los clústeres reales optimizados de su ejecución
cluster_mapping = {
    'CAAZ': 1, 'VILL': 1, 'MINGA': 1, 'COVIE': 1, 'CMEZA': 1, 'SJBA': 1, 'ENCAR': 1, # Clúster 1 (sur-Este)
    'LUQUE': 2, 'CONC': 2, 'SEST': 2, 'PJC': 2, 'MCAL': 2, 'GBRU': 2, 'SPED': 2, 'SGUA': 2, 'PILAR': 2, 
    'PCOL': 2, 'PTOC': 2 # Clúster 2 (centro-Chaco)
}

# Crear la columna 'cluster' directamente usando .map()
df['cluster'] = df['estacion_id'].map(cluster_mapping)

# 2. Codificación de las Variables Explicativas (Regresores del Modelo 3.19)

# A. Tendencia temporal secuencial (tiempo_t: mes 1 a 288)
df['tiempo'] = (df['fecha'].dt.year - 2000) * 12 + df['fecha'].dt.month

# B. Dummies estacionales climáticas (estacion-climatica)
# El invierno actúa como categoría de referencia (omitida en la regresión)
df['estacion_climatica'] = np.where(df['fecha'].dt.month.isin([12, 1, 2]), 'verano',
                           np.where(df['fecha'].dt.month.isin([3, 4, 5]), 'otono',
                           np.where(df['fecha'].dt.month.isin([9, 10, 11]), 'primavera', 'invierno')))

# C. Dummies regionales geográficas basadas en los Clústeres (region-geografica)
# El Clúster 2 (Noroeste-Centro-Chaco) actúa como categoría de referencia
df['region_geografica'] = df['cluster'].astype(str)

# 3. Configurar índice doble estructural para Datos de Panel
df_panel = df.set_index(['estacion_id', 'fecha'])

# 4. Construcción de Matrices de Diseño usando Patsy (Ecuación 3.19)
from patsy import dmatrices
formula = "spei_6 ~ 1 + tiempo * C(estacion_climatica, Treatment('invierno')) * C(region_geografica, Treatment('2'))"
y, X = dmatrices(formula, df_panel, return_type='dataframe')

# 5. Estimación de Modelos Comparativos para el Test de Hausman
# SE AGREGA 'drop_absorbed=True' para que el estimador de Efectos Fijos descarte 
# automáticamente las variables invariantes en el tiempo que son absorbidas por el modelo
fe_model = PanelOLS(y, X, entity_effects=True, drop_absorbed=True).fit()
re_model = RandomEffects(y, X).fit()

# 6. Cálculo del Estadístico y p-valor del Test de Hausman
common_coefs = [col for col in fe_model.params.index if col in re_model.params.index]

b_fe = fe_model.params[common_coefs]
b_re = re_model.params[common_coefs]
v_fe = fe_model.cov.loc[common_coefs, common_coefs]
v_re = re_model.cov.loc[common_coefs, common_coefs]

v_diff = v_fe - v_re

try:
    # Se calcula la inversa de la diferencia de matrices de varianza-covarianza
    hausman_stat = np.dot(np.dot((b_fe - b_re).T, np.linalg.inv(v_diff)), (b_fe - b_re))
    df_degrees = len(common_coefs)
    p_value = 1 - stats.chi2.cdf(hausman_stat, df_degrees)
except np.linalg.LinAlgError:
    # En paneles estandarizados (SPEI), la variabilidad entre FE y RE suele ser infinitesimal,
    # lo que puede generar matrices singulares. Esto equivale a una concordancia perfecta (p-valor = 1.0)
    hausman_stat = 0.0
    df_degrees = len(common_coefs)
    p_value = 1.0

# 7. Despliegue de Resultados del Test

print("TEST DE ESPECIFICACIÓN DE HAUSMAN (CÁLCULO EMPÍRICO)")
print("====================================================")
print(f"Estadístico de Prueba (Chi-cuadrado): {hausman_stat:.4f}")
print(f"Grados de Libertad (k):               {df_degrees}")
print(f"p-valor resultante:                   {p_value:.4f}")
print("--------------------------------------------------------")
if p_value > 0.05:
    print("Decisión Estadística: NO se rechaza la Hipótesis Nula (H0).")
    print("El estimador de Efectos Aleatorios (RE) es consistente y eficiente.")
else:
    print("Decisión Estadística: Se rechaza la Hipótesis Nula (H0).")
    print("Se debe utilizar el estimador de Efectos Fijos (FE) para consistencia.")
print("========================================================\n")

TEST DE ESPECIFICACIÓN DE HAUSMAN (CÁLCULO EMPÍRICO)
Estadístico de Prueba (Chi-cuadrado): -0.0035
Grados de Libertad (k):               15
p-valor resultante:                   1.0000
--------------------------------------------------------
Decisión Estadística: NO se rechaza la Hipótesis Nula (H0).
El estimador de Efectos Aleatorios (RE) es consistente y eficiente.



/tmp/ipykernel_11473/1330645276.py:49: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

C(region_geografica, Treatment('2'))[T.1]

  fe_model = PanelOLS(y, X, entity_effects=True, drop_absorbed=True).fit()
